# M2 · Autograd

**Outcome:** Inspect the autograd lifecycle and control where gradients stop.

Run cells with **Shift + Enter**. PyTorch is already installed in standard Colab runtimes.

## Experiment question

> Which leaves will receive gradients, and which route will detach remove?

Before running code, write a prediction. Then observe the evidence, change one variable, and explain the difference.

In [ ]:
import torch
print('PyTorch', torch.__version__)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 1 · Record a forward trail
Only the values that may learn need `requires_grad=True`. Inspect `is_leaf` and `grad_fn` before calling backward.

In [ ]:
x = torch.tensor(2.)
w = torch.tensor(-1., requires_grad=True)
b = torch.tensor(.5, requires_grad=True)
loss = (x*w + b)**2
print('w is leaf:', w.is_leaf, 'w.grad:', w.grad)
print('loss is leaf:', loss.is_leaf, 'loss.grad_fn:', loss.grad_fn)

## 2 · Walk backward and clear the answers
The graph is recorded during forward; `.backward()` fills gradients on the leaves.

In [ ]:
loss.backward()
print('loss:', loss.item(), 'dw:', w.grad.item(), 'db:', b.grad.item())
w.grad.zero_(); b.grad.zero_()
print('after clearing:', w.grad.item(), b.grad.item())

## 3 · Detach one branch
The two modes produce the same forward values. Compare which parameters receive gradients.

In [ ]:
x = torch.tensor(3.)
teacher_w = torch.tensor(2., requires_grad=True)
student_w = torch.tensor(1., requires_grad=True)
teacher_prediction = teacher_w * x
target = teacher_prediction.detach()
student_prediction = student_w * x
loss = (student_prediction - target).pow(2)
loss.backward()
print('target:', target.item(), 'loss:', loss.item())
print('student grad:', student_w.grad.item())
print('teacher grad:', teacher_w.grad)  # None: detach cut this route

## Try it
Remove `.detach()`, rebuild the tensors, and run again. Predict the teacher gradient before printing it. Then wrap both model calls in `torch.no_grad()` and inspect `requires_grad` on their outputs.

## Reflection

1. What did you predict?
2. What evidence did the output provide?
3. Which one variable did you change?
4. How does the result connect to the lesson's mental model?